# 00 - Preparación, limpieza y depuración de la serie horaria de O3

Este primer notebook transforma el conjunto de datos bruto de O3 de la estación del distrito del Eixample de Barcelona en una serie temporal horaria preparada para el posterior análisis exploratorio (EDA) y modelado.

Los pasos que se seguirán en esta fase son los siguientes:

1. Cargar el archivo que contiene el conjunto de datos bruto.
2. Comprobar su estructura.
3. Transformar el formato diario ancho (cada registro cuenta con columnas `h01`-`h24`), en una serie horaria.
4. Comprobar la continuidad temporal, los duplicados, los valores ausentes y posibles valores inválidos.
5. Documentar los huecos existentes en la serie.
6. Guardar el dataset procesado y cuya obtención es reproducible en `data/processed/`.

Durante este proceso, el archivo que contiene el conjunto de datos en bruto no se modificará.

In [1]:
from pathlib import Path
import hashlib

import numpy as np
import pandas as pd
from IPython.display import display


# Se amplía la cantidad de columnas visibles para inspeccionar mejor las tablas.
pd.set_option("display.max_columns", 100)

# Se evita que pandas corte demasiado pronto las salidas.
pd.set_option("display.max_rows", 100)

In [2]:
def find_project_root(start: Path | None = None) -> Path:
    """Busca el directorio raíz del proyecto partiendo del actual."""
    start = Path.cwd() if start is None else start

    for candidate in [start, *start.parents]:
        if (candidate / "data").exists() and (candidate / "src").exists():
            return candidate

    raise FileNotFoundError("No se ha podido localizar el directorio raíz del proyecto.")


# Se localiza automáticamente el directorio raíz del repositorio.
PROJECT_ROOT = find_project_root()

# Conjunto de datos bruto utilizado en el TFG.
RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "Eixample_O3_2020_2025.csv"

# Archivo procesado que se generará a partir del CSV bruto.
PROCESSED_DATA_PATH = PROJECT_ROOT / "data" / "processed" / "o3_hourly.parquet"

# README correspondiente al directorio donde se almacenarán los datos procesados.
PROCESSED_README_PATH = PROJECT_ROOT / "data" / "processed" / "README.md"

# Directorio donde se guardarán las de tablas de resultados.
TABLES_DIR = PROJECT_ROOT / "reports" / "tables"

# Se comprueba que los directorios de salida existan.
PROCESSED_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Raíz del proyecto: {PROJECT_ROOT}")
print(f"Archivo bruto: {RAW_DATA_PATH}")
print(f"Archivo procesado: {PROCESSED_DATA_PATH}")
print(f"README procesados: {PROCESSED_README_PATH}")

Raíz del proyecto: C:\trabajo_github
Archivo bruto: C:\trabajo_github\data\raw\Eixample_O3_2020_2025.csv
Archivo procesado: C:\trabajo_github\data\processed\o3_hourly.parquet
README procesados: C:\trabajo_github\data\processed\README.md


In [3]:
def compute_file_sha256(file_path: Path) -> str:
    """Calcula el hash SHA256 de un archivo."""
    sha256 = hashlib.sha256()

    with file_path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            sha256.update(chunk)

    return sha256.hexdigest().upper()

# Se carga el archivo con el conjunto de datos en bruto sin aplicar, todavía, ninguna transformación.
raw_df = pd.read_csv(RAW_DATA_PATH)

# El hash permite identificar de forma inequívoca la versión del conjunto de datos bruto utilizado.
raw_file_sha256 = compute_file_sha256(RAW_DATA_PATH)

print(f"Filas del CSV bruto: {raw_df.shape[0]:,}")
print(f"Columnas del CSV bruto: {raw_df.shape[1]:,}")
print(f"SHA256 del CSV bruto: {raw_file_sha256}")

display(raw_df.head())
display(raw_df.tail())

Filas del CSV bruto: 2,192
Columnas del CSV bruto: 40
SHA256 del CSV bruto: 2C80695EB869DB2360F6C44FA81C1F2ABC9F83A217688FBAC9B498EA3059F276


,codi_eoi,nom_estacio,data,magnitud,contaminant,unitats,tipus_estacio,area_urbana,codi_ine,municipi,codi_comarca,nom_comarca,h01,h02,h03,h04,h05,h06,h07,h08,h09,h10,h11,h12,h13,h14,h15,h16,h17,h18,h19,h20,h21,h22,h23,h24,altitud,latitud,longitud,geocoded_column
0,8019043,Barcelona (Eixample),2025-12-31T00:00:00.000,14,O3,µg/m3,traffic,urban,8019,Barcelona,13,Barcelonès,12.0,34.0,28.0,29.0,22.0,26.0,31.0,29.0,26.0,23.0,28.0,30.0,39.0,43.0,35.0,36.0,24.0,23.0,29.0,25.0,32.0,36.0,39.0,42.0,26,41.385315,2.1538,NaN
1,8019043,Barcelona (Eixample),2025-12-30T00:00:00.000,14,O3,µg/m3,traffic,urban,8019,Barcelona,13,Barcelonès,10.0,21.0,28.0,27.0,29.0,37.0,39.0,30.0,33.0,36.0,36.0,34.0,30.0,29.0,35.0,39.0,34.0,34.0,37.0,37.0,27.0,6.0,4.0,14.0,26,41.385315,2.1538,NaN
2,8019043,Barcelona (Eixample),2025-12-29T00:00:00.000,14,O3,µg/m3,traffic,urban,8019,Barcelona,13,Barcelonès,28.0,35.0,36.0,45.0,54.0,53.0,42.0,16.0,15.0,NaN,NaN,NaN,29.0,30.0,46.0,46.0,34.0,26.0,19.0,5.0,3.0,3.0,3.0,3.0,26,41.385315,2.1538,NaN
3,8019043,Barcelona (Eixample),2025-12-28T00:00:00.000,14,O3,µg/m3,traffic,urban,8019,Barcelona,13,Barcelonès,64.0,64.0,53.0,61.0,50.0,54.0,51.0,46.0,40.0,41.0,51.0,65.0,71.0,77.0,73.0,71.0,70.0,68.0,60.0,41.0,27.0,30.0,32.0,33.0,26,41.385315,2.1538,NaN
4,8019043,Barcelona (Eixample),2025-12-27T00:00:00.000,14,O3,µg/m3,traffic,urban,8019,Barcelona,13,Barcelonès,67.0,67.0,74.0,75.0,74.0,63.0,73.0,76.0,77.0,80.0,72.0,58.0,73.0,67.0,70.0,73.0,73.0,75.0,72.0,66.0,68.0,73.0,71.0,69.0,26,41.385315,2.1538,NaN


,codi_eoi,nom_estacio,data,magnitud,contaminant,unitats,tipus_estacio,area_urbana,codi_ine,municipi,codi_comarca,nom_comarca,h01,h02,h03,h04,h05,h06,h07,h08,h09,h10,h11,h12,h13,h14,h15,h16,h17,h18,h19,h20,h21,h22,h23,h24,altitud,latitud,longitud,geocoded_column
2187,8019043,Barcelona (Eixample),2020-01-05T00:00:00.000,14,O3,µg/m3,traffic,urban,8019,Barcelona,13,Barcelonès,31.0,15.0,19.0,31.0,29.0,24.0,22.0,19.0,8.0,11.0,8.0,8.0,10.0,4.0,12.0,33.0,29.0,22.0,9.0,1.0,1.0,1.0,2.0,3.0,26,41.385315,2.1538,POINT (2.1537998 41.385315)
2188,8019043,Barcelona (Eixample),2020-01-04T00:00:00.000,14,O3,µg/m3,traffic,urban,8019,Barcelona,13,Barcelonès,1.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,2.0,2.0,5.0,11.0,14.0,28.0,28.0,19.0,23.0,27.0,25.0,22.0,26.0,8.0,15.0,26,41.385315,2.1538,POINT (2.1537998 41.385315)
2189,8019043,Barcelona (Eixample),2020-01-03T00:00:00.000,14,O3,µg/m3,traffic,urban,8019,Barcelona,13,Barcelonès,1.0,17.0,35.0,33.0,25.0,28.0,17.0,11.0,9.0,10.0,7.0,11.0,11.0,11.0,10.0,6.0,4.0,2.0,1.0,1.0,1.0,1.0,2.0,1.0,26,41.385315,2.1538,POINT (2.1537998 41.385315)
2190,8019043,Barcelona (Eixample),2020-01-02T00:00:00.000,14,O3,µg/m3,traffic,urban,8019,Barcelona,13,Barcelonès,4.0,5.0,5.0,4.0,1.0,1.0,2.0,1.0,2.0,4.0,9.0,22.0,31.0,33.0,33.0,34.0,37.0,22.0,2.0,2.0,2.0,5.0,9.0,1.0,26,41.385315,2.1538,POINT (2.1537998 41.385315)
2191,8019043,Barcelona (Eixample),2020-01-01T00:00:00.000,14,O3,µg/m3,traffic,urban,8019,Barcelona,13,Barcelonès,22.0,19.0,19.0,6.0,8.0,5.0,4.0,2.0,2.0,8.0,13.0,25.0,27.0,19.0,22.0,25.0,26.0,18.0,16.0,15.0,13.0,11.0,3.0,1.0,26,41.385315,2.1538,POINT (2.1537998 41.385315)


In [4]:
# Las columnas h01-h24 contienen las mediciones horarias correspondientes a día.
hour_columns = [f"h{hour:02d}" for hour in range(1, 25)]

# Definimos, como buena práctica, las columnas mínimas que se espera encontrar en el conjunto de datos bruto.
required_columns = [
    "data",
    "nom_estacio",
    "contaminant",
    "unitats",
    *hour_columns,
]

missing_required_columns = [
    column for column in required_columns
    if column not in raw_df.columns
]

if missing_required_columns:
    raise ValueError(f"Faltan columnas obligatorias: {missing_required_columns}")

# El resto de las columnas se considerarán como metadatos relativos a la estación o la ubicación de esta.
metadata_columns = [
    column for column in raw_df.columns
    if column not in hour_columns
]

print("Columnas horarias detectadas:")
print(hour_columns)

print("\nColumnas de metadatos detectadas:")
print(metadata_columns)

# Se comprueba que el archivo contenga una única estación, tipo de contaminante y unidad de medición.
metadata_summary = raw_df[
    ["nom_estacio", "contaminant", "unitats", "municipi"]
].drop_duplicates()

display(metadata_summary)

if raw_df["contaminant"].nunique() != 1:
    raise ValueError("El CSV contiene más de un contaminante.")

if raw_df["nom_estacio"].nunique() != 1:
    raise ValueError("El CSV contiene más de una estación.")

Columnas horarias detectadas:
['h01', 'h02', 'h03', 'h04', 'h05', 'h06', 'h07', 'h08', 'h09', 'h10', 'h11', 'h12', 'h13', 'h14', 'h15', 'h16', 'h17', 'h18', 'h19', 'h20', 'h21', 'h22', 'h23', 'h24']

Columnas de metadatos detectadas:
['codi_eoi', 'nom_estacio', 'data', 'magnitud', 'contaminant', 'unitats', 'tipus_estacio', 'area_urbana', 'codi_ine', 'municipi', 'codi_comarca', 'nom_comarca', 'altitud', 'latitud', 'longitud', 'geocoded_column']


,nom_estacio,contaminant,unitats,municipi
0,Barcelona (Eixample),O3,µg/m3,Barcelona


In [5]:
def convert_daily_wide_to_hourly(
    raw_data: pd.DataFrame,
    hour_columns: list[str],
) -> pd.DataFrame:
    """Convierte el formato diario ancho con formato h01-h24 en una serie horaria.

    La convención adoptada será la siguiente:
    - h01 se corresponderá con las 00:00 horas,
    - h02 se corresponderá con las 01:00 horas,
    - ...
    - h24 se corresponderá con las 23:00 horas.

    Esta convención permitirá construir una serie horaria regular a partir del conjunto de datos bruto.
    """
    data = raw_data.copy()

    # La columna "data" representa el día de observación.
    data["date"] = pd.to_datetime(data["data"]).dt.normalize()

    # Se pasa de 24 columnas horarias a una fila para cada hora.
    long_data = data.melt(
        id_vars=[column for column in data.columns if column not in hour_columns],
        value_vars=hour_columns,
        var_name="hour_label",
        value_name="o3",
    )

    # La columna h01 se transforma en la hora 00:00, la h02 en la hora 01:00, etc.
    long_data["hour"] = (
        long_data["hour_label"]
        .str.replace("h", "", regex=False)
        .astype(int)
        - 1
    )

    # Se construye la marca temporal horaria (es decir, el timestamp) a partir del día y la hora.
    long_data["timestamp"] = (
        long_data["date"] + pd.to_timedelta(long_data["hour"], unit="h")
    )

    # Se trasladan los valores del ozono troposférico (O3) a formato numérico.
    # Si no es posible convertir algún valor, se registra como NaN.
    long_data["o3"] = pd.to_numeric(long_data["o3"], errors="coerce")

    # Se conservan la variable objetivo (o3) y los metadatos.
    keep_columns = [
        "timestamp",
        "o3",
        "nom_estacio",
        "contaminant",
        "unitats",
        "codi_eoi",
        "municipi",
        "altitud",
        "latitud",
        "longitud",
    ]

    keep_columns = [column for column in keep_columns if column in long_data.columns]

    hourly_data = (
        long_data[keep_columns]
        .sort_values("timestamp")
        .reset_index(drop=True)
    )

    return hourly_data


hourly_df = convert_daily_wide_to_hourly(raw_df, hour_columns)

print(f"Filas horarias generadas: {hourly_df.shape[0]:,}")

display(hourly_df.head())
display(hourly_df.tail())

Filas horarias generadas: 52,608


,timestamp,o3,nom_estacio,contaminant,unitats,codi_eoi,municipi,altitud,latitud,longitud
0,2020-01-01 00:00:00,22.0,Barcelona (Eixample),O3,µg/m3,8019043,Barcelona,26,41.385315,2.1538
1,2020-01-01 01:00:00,19.0,Barcelona (Eixample),O3,µg/m3,8019043,Barcelona,26,41.385315,2.1538
2,2020-01-01 02:00:00,19.0,Barcelona (Eixample),O3,µg/m3,8019043,Barcelona,26,41.385315,2.1538
3,2020-01-01 03:00:00,6.0,Barcelona (Eixample),O3,µg/m3,8019043,Barcelona,26,41.385315,2.1538
4,2020-01-01 04:00:00,8.0,Barcelona (Eixample),O3,µg/m3,8019043,Barcelona,26,41.385315,2.1538


,timestamp,o3,nom_estacio,contaminant,unitats,codi_eoi,municipi,altitud,latitud,longitud
52603,2025-12-31 19:00:00,25.0,Barcelona (Eixample),O3,µg/m3,8019043,Barcelona,26,41.385315,2.1538
52604,2025-12-31 20:00:00,32.0,Barcelona (Eixample),O3,µg/m3,8019043,Barcelona,26,41.385315,2.1538
52605,2025-12-31 21:00:00,36.0,Barcelona (Eixample),O3,µg/m3,8019043,Barcelona,26,41.385315,2.1538
52606,2025-12-31 22:00:00,39.0,Barcelona (Eixample),O3,µg/m3,8019043,Barcelona,26,41.385315,2.1538
52607,2025-12-31 23:00:00,42.0,Barcelona (Eixample),O3,µg/m3,8019043,Barcelona,26,41.385315,2.1538


In [6]:
# Se construye el índice horario esperado (esto es, desde el 1 de enero de 2020
# hasta el 31 de diciembre de 2025) desde la primera hasta la última observación.
expected_index = pd.date_range(
    start=hourly_df["timestamp"].min(),
    end=hourly_df["timestamp"].max(),
    freq="h",
)

# Se buscan horas ausentes en la secuencia temporal.
missing_timestamps = expected_index.difference(hourly_df["timestamp"])

# Se comprueba si existe más de una observación para una misma hora.
duplicated_timestamps = hourly_df["timestamp"].duplicated().sum()

print(f"Inicio de la serie: {hourly_df['timestamp'].min()}")
print(f"Fin de la serie: {hourly_df['timestamp'].max()}")
print(f"Registros horarios esperados: {len(expected_index):,}")
print(f"Registros horarios generados: {len(hourly_df):,}")
print(f"Timestamps faltantes: {len(missing_timestamps):,}")
print(f"Timestamps duplicados: {duplicated_timestamps:,}")

if duplicated_timestamps > 0:
    raise ValueError("Se han detectado timestamps duplicados.")

# Se reindexa explícitamente para garantizar frecuencia horaria regular.
# En este dataset no deberían faltar timestamps, pero esta operación deja la
# regularidad temporal garantizada de cara a análisis posteriores.
hourly_regular_df = (
    hourly_df
    .set_index("timestamp")
    .reindex(expected_index)
    .rename_axis("timestamp")
    .reset_index()
)

display(hourly_regular_df.head())
display(hourly_regular_df.tail())

Inicio de la serie: 2020-01-01 00:00:00
Fin de la serie: 2025-12-31 23:00:00
Registros horarios esperados: 52,608
Registros horarios generados: 52,608
Timestamps faltantes: 0
Timestamps duplicados: 0


,timestamp,o3,nom_estacio,contaminant,unitats,codi_eoi,municipi,altitud,latitud,longitud
0,2020-01-01 00:00:00,22.0,Barcelona (Eixample),O3,µg/m3,8019043,Barcelona,26,41.385315,2.1538
1,2020-01-01 01:00:00,19.0,Barcelona (Eixample),O3,µg/m3,8019043,Barcelona,26,41.385315,2.1538
2,2020-01-01 02:00:00,19.0,Barcelona (Eixample),O3,µg/m3,8019043,Barcelona,26,41.385315,2.1538
3,2020-01-01 03:00:00,6.0,Barcelona (Eixample),O3,µg/m3,8019043,Barcelona,26,41.385315,2.1538
4,2020-01-01 04:00:00,8.0,Barcelona (Eixample),O3,µg/m3,8019043,Barcelona,26,41.385315,2.1538


,timestamp,o3,nom_estacio,contaminant,unitats,codi_eoi,municipi,altitud,latitud,longitud
52603,2025-12-31 19:00:00,25.0,Barcelona (Eixample),O3,µg/m3,8019043,Barcelona,26,41.385315,2.1538
52604,2025-12-31 20:00:00,32.0,Barcelona (Eixample),O3,µg/m3,8019043,Barcelona,26,41.385315,2.1538
52605,2025-12-31 21:00:00,36.0,Barcelona (Eixample),O3,µg/m3,8019043,Barcelona,26,41.385315,2.1538
52606,2025-12-31 22:00:00,39.0,Barcelona (Eixample),O3,µg/m3,8019043,Barcelona,26,41.385315,2.1538
52607,2025-12-31 23:00:00,42.0,Barcelona (Eixample),O3,µg/m3,8019043,Barcelona,26,41.385315,2.1538


In [7]:
# Se crea una copia final del dataset procesado.
hourly_processed_df = hourly_regular_df.copy()

# En una concentración de O3, los valores negativos no son físicamente válidos.
# Si aparecieran, se marcarían como ausentes.
invalid_negative_o3_mask = (
    hourly_processed_df["o3"].notna()
    & (hourly_processed_df["o3"] < 0)
)

invalid_negative_o3_values = int(invalid_negative_o3_mask.sum())

hourly_processed_df.loc[invalid_negative_o3_mask, "o3"] = np.nan

# No se eliminan valores altos automáticamente.
# En calidad del aire, los valores extremos pueden corresponder a episodios reales.
missing_o3_values = int(hourly_processed_df["o3"].isna().sum())

preprocessing_quality_summary = pd.DataFrame(
    {
        "metric": [
            "raw_daily_records",
            "raw_columns",
            "start_timestamp",
            "end_timestamp",
            "expected_hourly_records",
            "processed_hourly_records",
            "missing_timestamps",
            "duplicated_timestamps",
            "missing_o3_values",
            "missing_o3_percentage",
            "invalid_negative_o3_values",
            "min_o3",
            "max_o3",
            "mean_o3",
            "median_o3",
            "std_o3",
        ],
        "value": [
            raw_df.shape[0],
            raw_df.shape[1],
            hourly_processed_df["timestamp"].min(),
            hourly_processed_df["timestamp"].max(),
            len(expected_index),
            len(hourly_processed_df),
            len(missing_timestamps),
            duplicated_timestamps,
            missing_o3_values,
            missing_o3_values / len(hourly_processed_df) * 100,
            invalid_negative_o3_values,
            hourly_processed_df["o3"].min(),
            hourly_processed_df["o3"].max(),
            hourly_processed_df["o3"].mean(),
            hourly_processed_df["o3"].median(),
            hourly_processed_df["o3"].std(),
        ],
    }
)

display(preprocessing_quality_summary)

preprocessing_quality_summary.to_csv(
    TABLES_DIR / "preprocessing_quality_summary.csv",
    index=False,
)

,metric,value
0,raw_daily_records,2192
1,raw_columns,40
2,start_timestamp,2020-01-01 00:00:00
3,end_timestamp,2025-12-31 23:00:00
4,expected_hourly_records,52608
5,processed_hourly_records,52608
6,missing_timestamps,0
7,duplicated_timestamps,0
8,missing_o3_values,1666
9,missing_o3_percentage,3.166819


## Criterio de depuración adoptado

El dataset procesado generado por este notebook se considera la serie horaria base definitiva para el resto del proyecto.

La depuración se realiza con un criterio conservador:

1. El CSV bruto no se modifica.
2. El formato diario ancho `h01`-`h24` se transforma en una serie horaria regular.
3. Se comprueba que no existen timestamps duplicados ni huecos en el índice temporal.
4. La variable `o3` se convierte a formato numérico.
5. Los valores ausentes de `o3` se conservan como `NaN`.
6. No se imputa la variable objetivo en esta fase, para evitar introducir valores artificiales en el entrenamiento o la evaluación.
7. Los valores negativos, si aparecieran, se considerarían físicamente inválidos y se marcarían como ausentes.
8. Los valores altos no se eliminan automáticamente, ya que pueden corresponder a episodios reales de contaminación.
9. La posible imputación de huecos cortos se evaluará únicamente como análisis de sensibilidad en la fase de construcción de datasets supervisados.
10. Las muestras sin valor objetivo observado no se utilizarán como etiquetas de entrenamiento o evaluación.

In [8]:
# Se analiza la longitud de los huecos consecutivos de O3.
# No es lo mismo tener valores ausentes aislados que tener tramos largos sin observaciones.
missing_flags = (
    hourly_processed_df
    .set_index("timestamp")["o3"]
    .sort_index()
    .asfreq("h")
    .isna()
)

# Cada vez que cambia el estado observado/ausente, se crea un nuevo grupo.
run_id = missing_flags.ne(missing_flags.shift(fill_value=False)).cumsum()

missing_runs = (
    pd.DataFrame(
        {
            "is_missing": missing_flags,
            "run_id": run_id,
        }
    )
    .loc[lambda data: data["is_missing"]]
    .groupby("run_id")
    .agg(
        start_timestamp=("is_missing", lambda values: values.index.min()),
        end_timestamp=("is_missing", lambda values: values.index.max()),
        length_hours=("is_missing", "size"),
    )
    .reset_index(drop=True)
)

missing_run_summary = pd.Series(
    {
        "number_of_missing_runs": len(missing_runs),
        "max_missing_run_hours": missing_runs["length_hours"].max() if not missing_runs.empty else 0,
        "median_missing_run_hours": missing_runs["length_hours"].median() if not missing_runs.empty else 0,
        "mean_missing_run_hours": missing_runs["length_hours"].mean() if not missing_runs.empty else 0,
        "total_missing_hours": missing_runs["length_hours"].sum() if not missing_runs.empty else 0,
    },
    name="value",
)

print("Huecos consecutivos más largos:")
display(missing_runs.sort_values("length_hours", ascending=False).head(20))

print("Resumen de huecos:")
display(missing_run_summary.to_frame())

missing_runs.to_csv(TABLES_DIR / "preprocessing_missing_runs.csv", index=False)
missing_run_summary.to_csv(TABLES_DIR / "preprocessing_missing_run_summary.csv")

Huecos consecutivos más largos:


,start_timestamp,end_timestamp,length_hours
146,2024-01-04 12:00:00,2024-01-09 09:00:00,118
106,2022-11-03 17:00:00,2022-11-08 10:00:00,114
159,2024-05-31 12:00:00,2024-06-04 12:00:00,97
36,2021-01-29 05:00:00,2021-02-01 21:00:00,89
78,2022-04-08 03:00:00,2022-04-11 10:00:00,80
215,2025-11-14 11:00:00,2025-11-17 13:00:00,75
80,2022-04-30 10:00:00,2022-05-02 14:00:00,53
190,2025-05-21 10:00:00,2025-05-23 14:00:00,53
27,2020-11-14 08:00:00,2020-11-16 10:00:00,51
99,2022-09-11 16:00:00,2022-09-13 12:00:00,45


Resumen de huecos:


,value
number_of_missing_runs,218.000000
max_missing_run_hours,118.000000
median_missing_run_hours,2.000000
mean_missing_run_hours,7.642202
total_missing_hours,1666.000000


In [9]:
# Se resume la longitud de los huecos consecutivos por tramos.
# Esto ayuda a decidir si una imputación limitada de huecos cortos podría ser razonable
# en una fase posterior, sin rellenar todavía la variable objetivo.
missing_run_bins = pd.cut(
    missing_runs["length_hours"],
    bins=[0, 1, 2, 3, 6, 12, 24, 48, np.inf],
    labels=[
        "1 h",
        "2 h",
        "3 h",
        "4-6 h",
        "7-12 h",
        "13-24 h",
        "25-48 h",
        ">48 h",
    ],
    right=True,
)

missing_run_length_distribution = (
    missing_runs.assign(length_group=missing_run_bins)
    .groupby("length_group", observed=False)
    .agg(
        number_of_runs=("length_hours", "size"),
        total_missing_hours=("length_hours", "sum"),
    )
    .reset_index()
)

missing_run_length_distribution["missing_hours_percentage"] = (
    missing_run_length_distribution["total_missing_hours"]
    / missing_run_length_distribution["total_missing_hours"].sum()
    * 100
)

display(missing_run_length_distribution)

missing_run_length_distribution.to_csv(
    TABLES_DIR / "preprocessing_missing_run_length_distribution.csv",
    index=False,
)

,length_group,number_of_runs,total_missing_hours,missing_hours_percentage
0,1 h,66,66,3.961585
1,2 h,45,90,5.402161
2,3 h,30,90,5.402161
3,4-6 h,47,212,12.725090
4,7-12 h,5,41,2.460984
5,13-24 h,6,127,7.623049
6,25-48 h,10,310,18.607443
7,>48 h,9,730,43.817527


In [10]:
# Se analizan valores extremos de O3.
# No se eliminan automáticamente porque pueden corresponder a episodios reales
# de contaminación atmosférica.
observed_o3 = hourly_processed_df["o3"].dropna()

o3_distribution_thresholds = pd.Series(
    {
        "min": observed_o3.min(),
        "p01": observed_o3.quantile(0.01),
        "p05": observed_o3.quantile(0.05),
        "p25": observed_o3.quantile(0.25),
        "median": observed_o3.median(),
        "p75": observed_o3.quantile(0.75),
        "p95": observed_o3.quantile(0.95),
        "p99": observed_o3.quantile(0.99),
        "max": observed_o3.max(),
        "iqr": observed_o3.quantile(0.75) - observed_o3.quantile(0.25),
    },
    name="value",
)

display(o3_distribution_thresholds.to_frame())

o3_distribution_thresholds.to_csv(
    TABLES_DIR / "preprocessing_o3_distribution_thresholds.csv"
)

,value
min,1.0
p01,1.0
p05,2.0
p25,27.0
median,47.0
p75,64.0
p95,86.0
p99,102.0
max,167.0
iqr,37.0


In [11]:
# Se listan los valores más altos observados.
# El objetivo es revisar si los máximos parecen concentrarse en periodos concretos
# o si son episodios aislados.
top_o3_values = (
    hourly_processed_df
    .dropna(subset=["o3"])
    .sort_values("o3", ascending=False)
    .head(30)
)

display(top_o3_values)

top_o3_values.to_csv(
    TABLES_DIR / "preprocessing_top_o3_values.csv",
    index=False,
)

,timestamp,o3,nom_estacio,contaminant,unitats,codi_eoi,municipi,altitud,latitud,longitud
48230,2025-07-02 14:00:00,167.0,Barcelona (Eixample),O3,µg/m3,8019043,Barcelona,26,41.385315,2.1538
48229,2025-07-02 13:00:00,152.0,Barcelona (Eixample),O3,µg/m3,8019043,Barcelona,26,41.385315,2.1538
48231,2025-07-02 15:00:00,147.0,Barcelona (Eixample),O3,µg/m3,8019043,Barcelona,26,41.385315,2.1538
47897,2025-06-18 17:00:00,146.0,Barcelona (Eixample),O3,µg/m3,8019043,Barcelona,26,41.385315,2.1538
47896,2025-06-18 16:00:00,141.0,Barcelona (Eixample),O3,µg/m3,8019043,Barcelona,26,41.385315,2.1538
21589,2022-06-18 13:00:00,141.0,Barcelona (Eixample),O3,µg/m3,8019043,Barcelona,26,41.385315,2.1538
49168,2025-08-10 16:00:00,141.0,Barcelona (Eixample),O3,µg/m3,8019043,Barcelona,26,41.385315,2.1538
49311,2025-08-16 15:00:00,141.0,Barcelona (Eixample),O3,µg/m3,8019043,Barcelona,26,41.385315,2.1538
49334,2025-08-17 14:00:00,141.0,Barcelona (Eixample),O3,µg/m3,8019043,Barcelona,26,41.385315,2.1538
21588,2022-06-18 12:00:00,140.0,Barcelona (Eixample),O3,µg/m3,8019043,Barcelona,26,41.385315,2.1538


In [12]:
# Se analizan saltos horarios bruscos.
# Un cambio muy grande de una hora a la siguiente puede ser real,
# pero también puede indicar un posible problema de medición.
# Por eso se documenta en vez de eliminarlo automáticamente.
jump_diagnostic_df = hourly_processed_df[["timestamp", "o3"]].copy()

jump_diagnostic_df["o3_diff_1h"] = (
    jump_diagnostic_df["o3"] - jump_diagnostic_df["o3"].shift(1)
)

jump_diagnostic_df["abs_o3_diff_1h"] = jump_diagnostic_df["o3_diff_1h"].abs()

top_o3_hourly_jumps = (
    jump_diagnostic_df
    .dropna(subset=["abs_o3_diff_1h"])
    .sort_values("abs_o3_diff_1h", ascending=False)
    .head(30)
)

display(top_o3_hourly_jumps)

top_o3_hourly_jumps.to_csv(
    TABLES_DIR / "preprocessing_top_o3_hourly_jumps.csv",
    index=False,
)

,timestamp,o3,o3_diff_1h,abs_o3_diff_1h
49164,2025-08-10 12:00:00,93.0,80.0,80.0
31932,2023-08-23 12:00:00,111.0,67.0,67.0
38167,2024-05-09 07:00:00,15.0,-64.0,64.0
41301,2024-09-16 21:00:00,24.0,-62.0,62.0
22116,2022-07-10 12:00:00,97.0,61.0,61.0
49306,2025-08-16 10:00:00,112.0,59.0,59.0
49219,2025-08-12 19:00:00,20.0,-59.0,59.0
47040,2025-05-14 00:00:00,92.0,55.0,55.0
38246,2024-05-12 14:00:00,75.0,54.0,54.0
48297,2025-07-05 09:00:00,102.0,54.0,54.0


## Criterio sobre valores extremos

No se eliminan valores altos de O3 únicamente por ser extremos estadísticos. En una serie ambiental, las concentraciones elevadas pueden corresponder a episodios reales de contaminación.

El criterio adoptado es el siguiente:

- Los valores negativos de O3 se consideran físicamente inválidos y se marcarían como ausentes.
- Los valores altos se conservan salvo evidencia clara de error instrumental o de codificación.
- Los saltos horarios bruscos se documentan, pero no se eliminan automáticamente.
- La interpretación de episodios elevados se realizará en la EDA y en la evaluación posterior de los modelos.

Este criterio evita eliminar precisamente los episodios que podrían resultar más relevantes para evaluar la utilidad predictiva de los modelos.

In [13]:
# Se definen las columnas base que formarán parte del dataset procesado.
# Las columnas diagnósticas, como diferencias horarias o flags auxiliares,
# no se guardan en el parquet principal.
processed_columns = [
    "timestamp",
    "o3",
    "nom_estacio",
    "contaminant",
    "unitats",
    "codi_eoi",
    "municipi",
    "altitud",
    "latitud",
    "longitud",
]

processed_columns = [
    column for column in processed_columns
    if column in hourly_processed_df.columns
]

processed_base_df = hourly_processed_df[processed_columns].copy()

# Se guarda el dataset procesado en formato Parquet.
# Este archivo no se versiona en Git porque puede regenerarse ejecutando este notebook.
processed_base_df.to_parquet(PROCESSED_DATA_PATH, index=False)

print(f"Dataset procesado guardado en: {PROCESSED_DATA_PATH}")
print(f"Filas guardadas: {len(processed_base_df):,}")
print(f"Columnas guardadas: {list(processed_base_df.columns)}")

Dataset procesado guardado en: C:\trabajo_github\data\processed\o3_hourly.parquet
Filas guardadas: 52,608
Columnas guardadas: ['timestamp', 'o3', 'nom_estacio', 'contaminant', 'unitats', 'codi_eoi', 'municipi', 'altitud', 'latitud', 'longitud']


In [14]:
processed_readme = f"""# Datos procesados

Esta carpeta contiene archivos generados a partir del dataset bruto de O3.

## Archivo principal generado localmente

`o3_hourly.parquet`

Este archivo se obtiene ejecutando:

`notebooks/00_data_preparation_o3.ipynb`

## Descripción

El archivo `o3_hourly.parquet` contiene la serie horaria base de O3 transformada desde el formato diario ancho `h01`-`h24` del CSV bruto.

La variable objetivo se conserva en la columna `o3`, expresada en las unidades originales del archivo fuente.

## Decisiones de depuración

- El CSV bruto no se modifica.
- La serie se transforma a frecuencia horaria regular.
- Se comprueba continuidad temporal y ausencia de duplicados.
- Los valores no convertibles a número se consideran ausentes.
- Los valores negativos de O3, si aparecieran, se marcarían como ausentes.
- Los valores ausentes de O3 se conservan como `NaN`.
- No se imputa la variable objetivo en esta fase.
- No se eliminan valores altos automáticamente, ya que pueden corresponder a episodios reales.
- La posible imputación de huecos cortos se evaluará únicamente como análisis de sensibilidad en la fase de construcción de datasets supervisados.

## Salida de esta fase

El archivo `o3_hourly.parquet` se considera el dataset horario base definitivo para la EDA y la posterior ingeniería de características.

## Reproducibilidad

El archivo procesado no se versiona porque puede regenerarse a partir de:

- `data/raw/Eixample_O3_2020_2025.csv`
- `notebooks/00_data_preparation_o3.ipynb`

Hash SHA256 del archivo bruto usado en esta ejecución:

`{raw_file_sha256}`
"""

PROCESSED_README_PATH.write_text(processed_readme, encoding="utf-8")

print(f"README de datos procesados guardado en: {PROCESSED_README_PATH}")

README de datos procesados guardado en: C:\trabajo_github\data\processed\README.md


In [15]:
# Se recarga el archivo procesado para comprobar que se ha guardado correctamente.
loaded_processed_df = pd.read_parquet(PROCESSED_DATA_PATH)

print(f"Filas recargadas:    {loaded_processed_df.shape[0]:,}")
print(f"Columnas recargadas: {loaded_processed_df.shape[1]:,}")

display(loaded_processed_df.head())
display(loaded_processed_df.tail())

# Contrato mínimo del dataset procesado.
required_processed_columns = [
    "timestamp",
    "o3",
    "nom_estacio",
    "contaminant",
    "unitats",
]

missing_processed_columns = [
    column for column in required_processed_columns
    if column not in loaded_processed_df.columns
]

if missing_processed_columns:
    raise ValueError(f"Faltan columnas en el dataset procesado: {missing_processed_columns}")

if loaded_processed_df["timestamp"].duplicated().any():
    raise ValueError("El dataset procesado contiene timestamps duplicados.")

timestamp_diffs = loaded_processed_df["timestamp"].sort_values().diff().dropna()

if not (timestamp_diffs == pd.Timedelta(hours=1)).all():
    raise ValueError("El dataset procesado no tiene frecuencia horaria regular.")

if (loaded_processed_df["o3"].dropna() < 0).any():
    raise ValueError("El dataset procesado contiene valores negativos de O3.")

assert loaded_processed_df.shape == processed_base_df.shape
assert loaded_processed_df["timestamp"].min() == processed_base_df["timestamp"].min()
assert loaded_processed_df["timestamp"].max() == processed_base_df["timestamp"].max()

print("Contrato del dataset procesado verificado correctamente.")

Filas recargadas:    52,608


Columnas recargadas: 10


,timestamp,o3,nom_estacio,contaminant,unitats,codi_eoi,municipi,altitud,latitud,longitud
0,2020-01-01 00:00:00,22.0,Barcelona (Eixample),O3,µg/m3,8019043,Barcelona,26,41.385315,2.1538
1,2020-01-01 01:00:00,19.0,Barcelona (Eixample),O3,µg/m3,8019043,Barcelona,26,41.385315,2.1538
2,2020-01-01 02:00:00,19.0,Barcelona (Eixample),O3,µg/m3,8019043,Barcelona,26,41.385315,2.1538
3,2020-01-01 03:00:00,6.0,Barcelona (Eixample),O3,µg/m3,8019043,Barcelona,26,41.385315,2.1538
4,2020-01-01 04:00:00,8.0,Barcelona (Eixample),O3,µg/m3,8019043,Barcelona,26,41.385315,2.1538


,timestamp,o3,nom_estacio,contaminant,unitats,codi_eoi,municipi,altitud,latitud,longitud
52603,2025-12-31 19:00:00,25.0,Barcelona (Eixample),O3,µg/m3,8019043,Barcelona,26,41.385315,2.1538
52604,2025-12-31 20:00:00,32.0,Barcelona (Eixample),O3,µg/m3,8019043,Barcelona,26,41.385315,2.1538
52605,2025-12-31 21:00:00,36.0,Barcelona (Eixample),O3,µg/m3,8019043,Barcelona,26,41.385315,2.1538
52606,2025-12-31 22:00:00,39.0,Barcelona (Eixample),O3,µg/m3,8019043,Barcelona,26,41.385315,2.1538
52607,2025-12-31 23:00:00,42.0,Barcelona (Eixample),O3,µg/m3,8019043,Barcelona,26,41.385315,2.1538


Contrato del dataset procesado verificado correctamente.
